# Live Replay Runner (Offline)

This notebook replays historical bars through the **same live trading stack** used by `ml_intraday_v3/live_trading/live_runner.py`:

- `LiveFeatureGenerator` (feature parity)
- `LiveModelPredictor` (bundle inference)
- `LiveExecutionEngine` (stop/target sizing + position lifecycle)
- `MetricsTracker`-compatible metrics (equity, daily PnL, drawdown)

**No broker calls** are made. A `MockBroker` marks open positions to market and provides live-like equity/PnL.


In [ ]:
# Parameters (repo-root safe)
import sys
from pathlib import Path

# Find repo root regardless of the notebook's working directory
cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "ml_intraday_v3").is_dir() and (p / "runs").is_dir():
        repo_root = p
        break
if repo_root is None:
    raise RuntimeError(f"Could not locate repo root from cwd={cwd}")

# Ensure imports work even if kernel starts elsewhere
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

RUN_DIR = repo_root / "runs" / "v3_2022_5m"
CONFIG_DIR = repo_root / "ml_intraday_v3" / "configs"
BAR_SIZE = "5m"

# Choose a short window first (you can expand)
START = "2022-06-01"
END = "2022-06-02"
MAX_BARS = 500  # set None for no cap

# Optional: persist metrics/trades to disk
OUTPUT_DIR = repo_root / "analysis" / "_live_replay_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("repo_root:", repo_root)
print("RUN_DIR:", RUN_DIR)
print("CONFIG_DIR:", CONFIG_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


repo_root: /Users/eshaanganguly/Documents/projects/algos 3 topstep
RUN_DIR: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/v3_2022_5m
CONFIG_DIR: /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs
OUTPUT_DIR: /Users/eshaanganguly/Documents/projects/algos 3 topstep/analysis/_live_replay_outputs


In [ ]:
# Inspect available bars range
bars_path = RUN_DIR / f"bar_size={BAR_SIZE}" / "bars.parquet"
bars_meta = pd.read_parquet(bars_path)
print("Rows:", len(bars_meta))
print("Min ts:", bars_meta.index.min())
print("Max ts:", bars_meta.index.max())


In [ ]:
# Notebook bootstrap: make repo importable (fixes `ModuleNotFoundError: ml_intraday_v3`)
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "ml_intraday_v3").is_dir():
        repo_root = p
        break

if repo_root is None:
    raise RuntimeError(f"Could not locate repo root from cwd={cwd}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)
print("sys.path[0]:", sys.path[0])


Repo root: /Users/eshaanganguly/Documents/projects/algos 3 topstep
sys.path[0]: /Users/eshaanganguly/Documents/projects/algos 3 topstep


In [ ]:
import pandas as pd

from ml_intraday_v3.live_trading.replay import replay_session

artifacts = replay_session(
    run_dir=RUN_DIR,
    config_dir=CONFIG_DIR,
    bar_size=BAR_SIZE,
    start=START,
    end=END,
    max_bars=MAX_BARS,
    output_dir=OUTPUT_DIR,
)

metrics = artifacts.metrics
trade_log = artifacts.trade_log

metrics.tail(3), trade_log.tail(5)


ValueError: No bars available after slicing; adjust start/end/max_bars.

In [ ]:
# Quick plots
import matplotlib.pyplot as plt

# Metrics timestamps are stored as python datetimes; normalize for plotting
metrics_plot = metrics.copy()
metrics_plot["timestamp"] = pd.to_datetime(metrics_plot["timestamp"])

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
axes[0].plot(metrics_plot["timestamp"], metrics_plot["current_equity"], label="Equity")
axes[0].set_title("Equity")
axes[0].grid(True)

axes[1].plot(metrics_plot["timestamp"], metrics_plot["daily_pnl"], label="Daily PnL")
axes[1].set_title("Daily PnL")
axes[1].grid(True)

axes[2].plot(metrics_plot["timestamp"], metrics_plot["max_drawdown"], label="Max Drawdown")
axes[2].set_title("Max Drawdown")
axes[2].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Export artifacts
out_metrics = OUTPUT_DIR / "replay_metrics.parquet"
out_trades = OUTPUT_DIR / "replay_trade_log.parquet"

metrics.to_parquet(out_metrics, index=False)
trade_log.to_parquet(out_trades, index=False)

print("Wrote:", out_metrics)
print("Wrote:", out_trades)
